# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from itertools import product
from tqdm.notebook import tqdm

import gdown

In [19]:
url='https://drive.google.com/uc?id=1AlGvsJDSzPT_70caausx8bFuupIEZkfh&export=download'

output='day-of-week-not-scaled.csv'
gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1AlGvsJDSzPT_70caausx8bFuupIEZkfh&export=download
To: /home/mirshod/Desktop/DSB11_ML_Advanced.ID_886524-1/src/ex01/day-of-week-not-scaled.csv
100%|██████████| 286k/286k [00:01<00:00, 274kB/s]


'day-of-week-not-scaled.csv'

In [20]:
url2='https://drive.google.com/uc?id=1PELTt93UDt-yp5xOECdlj9uy7kBMBKd4&export=download'
output='dayofweek.csv'
gdown.download(url2, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1PELTt93UDt-yp5xOECdlj9uy7kBMBKd4&export=download
To: /home/mirshod/Desktop/DSB11_ML_Advanced.ID_886524-1/src/ex01/dayofweek.csv
100%|██████████| 10.7k/10.7k [00:00<00:00, 266kB/s]


'dayofweek.csv'

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [21]:
df=pd.read_csv('day-of-week-not-scaled.csv')
df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [22]:
df2=pd.read_csv('dayofweek.csv')
df2.head()

,Unnamed: 0,dayofweek
0,0,4
1,1,4
2,2,4
3,3,4
4,4,4


In [23]:
df['dayofweek']=df2['dayofweek']
df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [24]:
y=df['dayofweek']<5
X=df.drop(columns=['dayofweek'])

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [26]:
def show_model_ranks(model):
  rank_test_score=model.cv_results_['rank_test_score']
  mean_test_score=model.cv_results_['mean_test_score']
  std_test_score=model.cv_results_['std_test_score']
  params=model.cv_results_['params']
  df=pd.DataFrame({'rank_test_score':rank_test_score, 'mean_test_score':mean_test_score, 'std_test_score':std_test_score, 'params':params})
  print(df.sort_values('rank_test_score'))

In [27]:
params={
    'kernel':['linear', 'rbf', 'sigmoid'],
    'C':[0.01, 0.1, 1, 1.5, 5, 10],
    'gamma':['scale', 'auto'],
    'class_weight':['balanced', None],
    'random_state':[21],
    'probability':[True],
}
grid_svm=GridSearchCV(SVC(), params, cv=5, n_jobs=-1)
grid_svm.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid'],
                         'probability': [True], 'random_state': [21]})

In [28]:
show_model_ranks(grid_svm)

    rank_test_score  mean_test_score  std_test_score  \
64                1         0.925794        0.023625   
70                2         0.924304        0.020113   
58                3         0.904285        0.017549   
52                4         0.902060        0.020595   
40                5         0.850145        0.011201   
..              ...              ...             ...   
17               67         0.370178        0.001199   
41               69         0.365717        0.011052   
53               69         0.365717        0.013097   
29               71         0.365714        0.009897   
65               72         0.362751        0.014489   

                                               params  
64  {'C': 10, 'class_weight': 'balanced', 'gamma':...  
70  {'C': 10, 'class_weight': None, 'gamma': 'auto...  
58  {'C': 5, 'class_weight': None, 'gamma': 'auto'...  
52  {'C': 5, 'class_weight': 'balanced', 'gamma': ...  
40  {'C': 1.5, 'class_weight': 'balanced', 'gam

## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [29]:
params={
    'max_depth':range(1, 50),
    'class_weight':['balanced', None],
    'criterion':['entropy', 'gini'],
}
grid_tree=GridSearchCV(DecisionTreeClassifier(random_state=21), params, cv=5, n_jobs=-1)
grid_tree.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 50)})

In [30]:
show_model_ranks(grid_tree)

     rank_test_score  mean_test_score  std_test_score  \
70                 1         0.942872        0.011181   
68                 2         0.942129        0.013638   
67                 3         0.941388        0.011869   
66                 4         0.941388        0.012973   
80                 5         0.940642        0.011556   
..               ...              ...             ...   
99               191         0.817480        0.019926   
98               193         0.789294        0.017891   
49               193         0.789294        0.017891   
0                193         0.789294        0.017891   
147              193         0.789294        0.017891   

                                                params  
70   {'class_weight': 'balanced', 'criterion': 'gin...  
68   {'class_weight': 'balanced', 'criterion': 'gin...  
67   {'class_weight': 'balanced', 'criterion': 'gin...  
66   {'class_weight': 'balanced', 'criterion': 'gin...  
80   {'class_weight': 'balance

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [31]:
params={
    'n_estimators':[5, 10, 50, 100],
    'max_depth':range(1, 50),
    'class_weight':['balanced', None],
    'criterion':['entropy', 'gini']
}
grid_forest=GridSearchCV(RandomForestClassifier(), params, cv=5, n_jobs=-1)
grid_forest.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 50),
                         'n_estimators': [5, 10, 50, 100]})

In [32]:
show_model_ranks(grid_forest)

     rank_test_score  mean_test_score  std_test_score  \
763                1         0.953253        0.007706   
711                2         0.952513        0.009252   
558                3         0.952510        0.009268   
533                4         0.951778        0.007788   
86                 5         0.951775        0.006672   
..               ...              ...             ...   
590              780         0.654316        0.020160   
589              781         0.650602        0.020517   
394              782         0.645405        0.011456   
395              783         0.636497        0.011261   
591              784         0.632799        0.008982   

                                                params  
763  {'class_weight': None, 'criterion': 'gini', 'm...  
711  {'class_weight': None, 'criterion': 'gini', 'm...  
558  {'class_weight': None, 'criterion': 'entropy',...  
533  {'class_weight': None, 'criterion': 'entropy',...  
86   {'class_weight': 'balance

## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [33]:
# def cross_val_score(model, X, y):
#   scores=cross_val_score(model, X, y, cv=5, n_jobs=-1)
#   return np.mean(scores), np.std(scores)

In [34]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

param_names = list(param_grid.keys())
param_combinations = list(product(*param_grid.values()))

results = []

for values in tqdm(param_combinations, desc="Grid Search"):
    params = dict(zip(param_names, values))

    model = RandomForestClassifier( random_state=42, n_jobs=-1, **params
    )

    scores = cross_val_score( model, X, y, cv=5, scoring="accuracy", n_jobs=-1
    )

    results.append({
        **params,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    })

df_results = pd.DataFrame(results).sort_values(by="mean_accuracy", ascending=False)

df_results.head(10)


Grid Search:   0%|          | 0/24 [00:00<?, ?it/s]

,n_estimators,max_depth,min_samples_split,min_samples_leaf,mean_accuracy,std_accuracy
6,100,10.0,5,1,0.739141,0.156047
18,200,10.0,5,1,0.736767,0.154205
16,200,10.0,2,1,0.736767,0.152874
4,100,10.0,2,1,0.735582,0.153570
17,200,10.0,2,2,0.735580,0.149699
19,200,10.0,5,2,0.733800,0.149725
7,100,10.0,5,2,0.733800,0.158000
5,100,10.0,2,2,0.726678,0.151096
3,100,NaN,5,2,0.722543,0.151336
9,100,20.0,2,2,0.721353,0.143032


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [35]:
best_model=RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1)
best_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_split=5)

In [36]:
acc_score=accuracy_score(y_test, best_model.predict(X_test))
print("Accuracy score:", acc_score)

Accuracy score: 0.9053254437869822
